# Matmul IV lab: build siboehm's SGEMM_CUDA on a T4 and run every kernel

**Runtime → Change runtime type → T4 GPU**, then run the cells in order (about 10 minutes, most of it kernel 1).

What you will measure: GFLOP/s for each of siboehm's FP32 matmul kernels (1–12) and for cuBLAS (kernel 0) at 4096×4096 on a Colab T4, the percentage of cuBLAS each one reaches, and whether each kernel's registers and shared memory fit the T4's limits. The last cells count 128-bit shared-memory loads (`LDS.128`) in each kernel's machine code, to test a guess about why kernels 7 and 8 were slower.

- Repo: https://github.com/siboehm/SGEMM_CUDA (MIT License, Copyright (c) 2023 Simon Boehm). Pinned to commit `5a7dcc5` (2025-09-02), the version this notebook was written against.
- Article: https://siboehm.com/articles/22/CUDA-MMM (the numbers in it are from an RTX A6000, not a T4).
- T4 reference points: 8.1 TFLOPS FP32 peak, 320 GB/s, 40 SMs, 64 KB shared memory and 1,024 threads per SM. Colab T4 clocks vary between sessions, so expect run-to-run differences of several percent.

This notebook was written without access to an NVIDIA GPU and has not been run by its author. If a cell fails, the error message is part of the exercise.

In [ ]:
!nvidia-smi

## 1. Clone the repo at a fixed commit and read its license

In [ ]:
!rm -rf SGEMM_CUDA && git clone -q https://github.com/siboehm/SGEMM_CUDA
%cd SGEMM_CUDA
!git checkout -q 5a7dcc513d951ba764d51bc9d587b3163f3a894d && git log -1 --format='%h %ci %s'
!head -3 LICENSE

## 2. Ask the GPU for its limits

A small CUDA program queries the device, so the limits come from your actual GPU rather than from a datasheet.

In [ ]:
%%writefile devinfo.cu
#include <cstdio>
#include <cstdlib>
#define CHECK(x) do { cudaError_t e = (x); if (e != cudaSuccess) { \
  printf("CUDA error %s at %s:%d\n", cudaGetErrorString(e), __FILE__, __LINE__); exit(1); } } while (0)
int main() {
  int dev = 0; cudaDeviceProp p;
  CHECK(cudaGetDevice(&dev));
  CHECK(cudaGetDeviceProperties(&p, dev));
  int clockKHz = 0;
  CHECK(cudaDeviceGetAttribute(&clockKHz, cudaDevAttrClockRate, dev));
  printf("name %s, compute capability %d.%d\n", p.name, p.major, p.minor);
  printf("SMs %d, max clock %.0f MHz\n", p.multiProcessorCount, clockKHz / 1000.0);
  printf("max threads per block %d, per SM %d\n", p.maxThreadsPerBlock, p.maxThreadsPerMultiProcessor);
  printf("registers per SM %d, per block %d\n", p.regsPerMultiprocessor, p.regsPerBlock);
  printf("shared memory per block (static) %zu B, per SM %zu B\n", p.sharedMemPerBlock, p.sharedMemPerMultiprocessor);
  // FP32 peak = SMs x 64 FP32 lanes per SM on Turing x 2 FLOPs per FMA x clock
  printf("FP32 peak at max clock: %.2f TFLOPS (64 FP32 lanes per SM assumed for Turing)\n",
         p.multiProcessorCount * 64 * 2 * (clockKHz * 1e3) / 1e12);
  return 0;
}

In [ ]:
!nvcc -O3 -arch=sm_75 -o devinfo devinfo.cu && ./devinfo

## 3. Build for sm_75

The README says: set `CUDA_COMPUTE_CAPABILITY` in `CMakeLists.txt` to your GPU's compute capability (the repo ships with 86, the A6000), then `mkdir build && cd build && cmake .. && cmake --build .`. The T4 is compute capability 7.5, so we change 86 to 75. We build only the `sgemm` target (the benchmark binary). If CMake fails on your Colab image, the fallback compiles the same two source files with `nvcc` directly.

In [ ]:
!sed -i 's/set(CUDA_COMPUTE_CAPABILITY 86)/set(CUDA_COMPUTE_CAPABILITY 75)/' CMakeLists.txt
!grep -n CUDA_COMPUTE_CAPABILITY CMakeLists.txt | head -2
!mkdir -p build && cd build && cmake -DCMAKE_BUILD_TYPE=Release -DCMAKE_CUDA_COMPILER=$(which nvcc) .. > cmake.log 2>&1; tail -3 cmake.log
!cd build && cmake --build . --target sgemm -j 4 2>&1 | tail -5

In [ ]:
import os, subprocess
if not os.path.exists("build/sgemm"):
    print("CMake build failed; falling back to plain nvcc (same sources, sm_75)")
    r = subprocess.run("nvcc -O3 -arch=sm_75 -std=c++17 -Isrc sgemm.cu src/runner.cu -lcublas -o build/sgemm",
                       shell=True, capture_output=True, text=True)
    print(r.stdout[-3000:], r.stderr[-3000:])
print("binary exists:", os.path.exists("build/sgemm"))

## 4. Does each kernel fit the T4?

`cuobjdump -res-usage` prints the registers per thread (REG) and static shared memory per block (SHARED) that the compiler assigned to every kernel. We compare them with the T4 limits from step 2 and with each kernel's block size (from `src/runner.cu`, A6000 settings, used unchanged here).

In [ ]:
import re, subprocess
names = {  # substring of the mangled kernel name -> (kernel number, threads per block at 4096^2)
    "sgemm_naive": (1, 1024), "sgemm_global_mem_coalesce": (2, 1024), "sgemm_shared_mem_block": (3, 1024),
    "sgemm1DBlocktiling": (4, 512), "sgemm2DBlocktiling": (5, 256), "sgemmVectorize": (6, 256),
    "sgemmResolveBankConflicts": (7, 256), "sgemmResolveBankExtraCol": (8, 256), "sgemmAutotuned": (9, 256),
    "sgemmWarptiling": (10, 128), "sgemmDoubleBuffering": (11, 256), "runSgemmDoubleBuffering2": (12, 128),
}
def kernel_of(fn):
    # longest match first, so sgemmDoubleBuffering does not swallow runSgemmDoubleBuffering2
    for key in sorted(names, key=len, reverse=True):
        if key in fn:
            return key
    return None

out = subprocess.run("cuobjdump -res-usage build/sgemm", shell=True, capture_output=True, text=True).stdout
REGS_PER_SM, SMEM_PER_BLOCK, SMEM_PER_SM, THREADS_PER_SM = 65536, 49152, 65536, 1024  # T4; check against step 2
rows, fn = {}, None
for line in out.splitlines():
    m = re.search(r"Function (\S+):", line)
    if m:
        fn = m.group(1); continue
    m = re.search(r"REG:(\d+).*?SHARED:(\d+)", line)
    if m and fn and kernel_of(fn):
        key = kernel_of(fn)
        # kernels 5-8 are compiled twice (BM=BN=64 for small sizes, BM=BN=128 for >= 128);
        # keep the 128 one, which is what runs at 4096. Template args appear as Li128E in the mangled name.
        if key not in rows or "Li128E" in fn:
            rows[key] = (int(m.group(1)), int(m.group(2)))
print(f"{'kernel':>6} {'threads':>7} {'regs/thr':>8} {'smem/block':>10}  blocks/SM by regs, smem, threads  fits?")
for key, (num, thr) in sorted(names.items(), key=lambda kv: kv[1][0]):
    if key not in rows:
        print(f"{num:>6}  not found in binary"); continue
    reg, smem = rows[key]
    by_regs = REGS_PER_SM // (reg * thr) if reg else 99
    by_smem = SMEM_PER_SM // smem if smem else 99
    by_thr = THREADS_PER_SM // thr
    fits = reg * thr <= REGS_PER_SM and smem <= SMEM_PER_BLOCK and thr <= 1024
    print(f"{num:>6} {thr:>7} {reg:>8} {smem:>10}  {by_regs:>4} {by_smem:>4} {by_thr:>4}   {'yes' if fits else 'NO: exceeds a T4 limit'}")
# Kernel 11 declares 2*(128*16 + 16*256) floats = 48 KB of shared memory: exactly the 48 KB static limit.

## 5. Run every kernel at 128 … 4096 and keep the 4096 result

`./build/sgemm <n>` checks kernel *n* against cuBLAS at each size (128 to 4096), then times 50 back-to-back launches with CUDA events and prints GFLOPS. Kernel 0 is cuBLAS itself (FP32, no tensor cores on a T4). A kernel that fails verification or hits a CUDA error stops early; we record that instead of a number.

In [ ]:
import re, subprocess, time
A6000 = {0: 23249.6, 1: 309.0, 2: 1986.5, 3: 2980.3, 4: 8474.7, 5: 15971.7, 6: 18237.3, 7: 16213.4,
         8: 16459.2, 9: 19721.0, 10: 21779.3, 11: 17278.3}  # siboehm README, RTX A6000, GFLOP/s
T4_PEAK = 8100.0  # GFLOP/s FP32, NVIDIA T4 datasheet
results, notes = {}, {}
for k in range(0, 13):
    t0 = time.time()
    r = subprocess.run(["./build/sgemm", str(k)], capture_output=True, text=True, timeout=1200)
    lines = r.stdout.splitlines()
    m = [re.search(r"performance: \(\s*([\d.]+)\) GFLOPS\. size: \(4096\)", l) for l in lines]
    m = [x for x in m if x]
    if m:
        results[k] = float(m[-1].group(1))
    else:
        notes[k] = (r.stdout + r.stderr).strip().splitlines()[-3:]
    print(f"kernel {k:>2}: {results.get(k, 'no 4096 result')}  ({time.time() - t0:.0f} s)")

In [ ]:
cub = results.get(0)
print(f"{'kernel':>6} {'T4 GFLOP/s':>11} {'% cuBLAS':>9} {'% T4 peak':>10} | {'A6000 GFLOP/s':>13} {'A6000 % cuBLAS':>15}")
for k in range(0, 13):
    t4 = results.get(k)
    a = A6000.get(k)
    t4s = f"{t4:11.1f}" if t4 else f"{'-':>11}"
    pc = f"{100 * t4 / cub:8.1f}%" if (t4 and cub) else f"{'-':>9}"
    pk = f"{100 * t4 / T4_PEAK:9.1f}%" if t4 else f"{'-':>10}"
    a_s = f"{a:13.1f}" if a else f"{'-':>13}"
    apc = f"{100 * a / A6000[0]:14.1f}%" if a else f"{'-':>15}"
    print(f"{k:>6} {t4s} {pc} {pk} | {a_s} {apc}")
for k, n in notes.items():
    print(f"\nkernel {k} did not report a 4096 result; last lines:\n  " + "\n  ".join(n))

## 6. Test a guess: did the bank-conflict fixes lose the 128-bit loads?

The article says kernels 7 and 8 "eliminate the conflicts but were overall still slower" and gives no reason. One guess (ours, not the article's): kernel 6 reads `Bs` with 128-bit `LDS.128` instructions; kernel 7 re-lays `Bs` so each thread's values sit 16 floats apart, and kernel 8 pads each row of `Bs` to 133 floats (532 bytes, not a multiple of 16), so neither can use 128-bit shared loads for `Bs`. Count them in the machine code (SASS). If kernels 7 and 8 show fewer 128-bit loads than kernel 6, the guess survives; it still would not prove that this is the whole reason.

In [ ]:
import re, subprocess
sass = subprocess.run("cuobjdump -sass build/sgemm", shell=True, capture_output=True, text=True).stdout
counts, fn = {}, None
for line in sass.splitlines():
    m = re.search(r"Function : (\S+)", line)
    if m:
        fn = m.group(1); continue
    key = kernel_of(fn) if fn else None
    if not key:
        continue
    for op in re.findall(r"\b(LDS(?:\.[A-Z0-9]+)*)\b", line):
        c = counts.setdefault((names[key][0], fn[:60]), [0, 0])
        c[1 if ".128" in op else 0] += 1
print(f"{'kernel':>6}  {'scalar LDS':>10} {'LDS.128':>8}  instantiation")
for (num, fname), (scalar, wide) in sorted(counts.items()):
    print(f"{num:>6}  {scalar:>10} {wide:>8}  {fname}")

## 7. Compare with the article (RTX A6000, 4096×4096, from the repo README)

| Kernel | A6000 GFLOP/s | % of cuBLAS | step vs previous |
|---|---:|---:|---:|
| 1: Naive | 309.0 | 1.3% | – |
| 2: GMEM coalescing | 1,986.5 | 8.5% | 6.43× |
| 3: SMEM caching | 2,980.3 | 12.8% | 1.50× |
| 4: 1D blocktiling | 8,474.7 | 36.5% | 2.84× |
| 5: 2D blocktiling | 15,971.7 | 68.7% | 1.88× |
| 6: Vectorized mem access | 18,237.3 | 78.4% | 1.14× |
| 7: Avoid bank conflicts (linearize), README only | 16,213.4 | 69.7% | 0.89× vs 6 |
| 8: Avoid bank conflicts (offset), README only | 16,459.2 | 70.8% | 0.90× vs 6 |
| 9: Autotuning | 19,721.0 | 84.8% | 1.08× vs 6 |
| 10: Warptiling | 21,779.3 | 93.7% | 1.10× |
| 11: Double buffering, README only | 17,278.3 | 74.3% | 0.79× vs 10 |
| 0: cuBLAS | 23,249.6 | 100% | 1.07× vs 10 |

Kernel 12 (a second double-buffering attempt using `cuda::memcpy_async` and barriers) is in the repo but not in the README table.

Things to notice in your T4 run:
- Kernels 9, 10 and 11 use block and tile sizes autotuned **for the A6000** (see `src/runner.cu`). The article's own point is that tuned parameters don't transfer between GPUs: on an A100 the A6000's best kernel-9 config reached 12 TFLOPs, against 12.6 for the A100's own best. Expect the T4 ranking to differ from the A6000 ranking.
- The T4 has 64 KB of shared memory per SM (the A6000 has 100 KB), so kernel 11's 48 KB block runs one block per SM.

## Try this
1. Autotune for the T4: `scripts/kernel_9_autotuner.sh` sweeps BK, TM, TN, BM and BN by editing `src/runner.cu` and rebuilding. First change `export DEVICE="2"` to `"0"` in the script, and shrink the value lists, because the full sweep rebuilds for every config and takes hours.
2. Swap in the commented "Settings for A100" block for kernel 10 in `src/runner.cu`, rebuild, and see whether the smaller A100 tile does better on the smaller T4.
3. Profile one kernel with Nsight Compute: `!ncu --set full -k regex:Warptiling -c 1 ./build/sgemm 10` (Colab may block the performance counters; if so, note the error and try on a rented GPU).